# 权限控制与资源归属

学习目标：根据当前用户、资源所有者和角色检查读取、修改与删除权限，并验证拒绝请求不会改变资源。

前置知识：身份认证、依赖注入、HTTP 状态码、Pydantic 模型与字典操作。

适用版本：FastAPI 0.141.1、Pydantic 2；示例在 Notebook 内用 TestClient 发送请求。

环境准备：[FastAPI 环境与运行入口](README.md)。

工作目录：content/Web与应用开发/FastAPI。按顺序运行本篇单元即可；三个用户和记录都由本篇创建，不需要账号、数据库或服务进程。

固定 Bearer 字符串仅用作本地教学输入，用来模拟已经识别出的用户，不是可部署的认证方案。

## 1 识别用户之后，还要检查资源归属

身份认证（authentication）确认“当前用户是谁”；授权（authorization）判断“这个用户能对这条资源做什么”。Alice 与 Bob 都能通过身份认证，并不意味着 Bob 可以读取 Alice 的记录。

先用一个比较表达式观察归属关系。owner_id 是服务端保存的所有者标识，user.id 是已识别用户的标识。请求中的记录编号只是查找条件，知道编号不代表有权访问。

In [1]:
from pydantic import BaseModel


class User(BaseModel):
    id: str
    role: str


users = {
    "alice": User(id="alice", role="user"),
    "bob": User(id="bob", role="user"),
    "admin": User(id="admin", role="admin"),
}
records = {
    1: {"id": 1, "owner_id": "alice", "title": "Alice 的笔记"},
    2: {"id": 2, "owner_id": "bob", "title": "Bob 的笔记"},
}

# 两个用户均已识别，但只有 Alice 是记录 1 的所有者。
for name in ("alice", "bob"):
    print(name, "是记录 1 的所有者：", users[name].id == records[1]["owner_id"])

alice 是记录 1 的所有者： True
bob 是记录 1 的所有者： False


## 2 用安全依赖取得当前用户

HTTPBearer 读取 Authorization 请求头，返回包含 scheme 和 credentials 的对象；它本身不会验证凭据是否属于某个用户。auto_error=False 让缺失凭据返回 None，下面由 get_current_user 统一产生 401，并携带 WWW-Authenticate: Bearer。

本例把三个虚构字符串映射到上面服务端定义的用户。实际应用在这个边界接入可信的身份验证结果；角色不能由请求中的 role 字段决定。Security 用来声明安全依赖，依赖中的判断仍需自己编写。

In [2]:
from typing import Annotated

from fastapi import FastAPI, HTTPException, Response, Security
from fastapi.security import HTTPAuthorizationCredentials, HTTPBearer
from fastapi.testclient import TestClient

app = FastAPI()
bearer = HTTPBearer(auto_error=False)
token_users = {f"demo-{name}": user for name, user in users.items()}
auth_headers = {
    name: {"Authorization": f"Bearer demo-{name}"} for name in users
}


def get_current_user(
    credential: Annotated[HTTPAuthorizationCredentials | None, Security(bearer)],
) -> User:
    user = token_users.get(credential.credentials) if credential else None
    if user is None:
        raise HTTPException(
            401, "需要有效身份", headers={"WWW-Authenticate": "Bearer"}
        )
    return user

C:\Users\ZHUANG\miniconda3\envs\hands-on-computing\Lib\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


先给当前用户添加一个观察入口。CurrentUser 是 Annotated 类型别名，后续路由复用这项依赖。测试只显示用户和状态码，不显示凭据。

In [3]:
CurrentUser = Annotated[User, Security(get_current_user)]


@app.get("/me")
def read_me(user: CurrentUser) -> User:
    return user


with TestClient(app) as client:
    for name in users:
        result = client.get("/me", headers=auth_headers[name])
        assert result.status_code == 200
        assert result.json()["id"] == name
        print(name, result.status_code, result.json())
    for headers in ({}, {"Authorization": "Bearer unknown-demo"}):
        denied = client.get("/me", headers=headers)
        assert denied.status_code == 401
        assert denied.headers["www-authenticate"] == "Bearer"
        print("身份未通过：", denied.status_code, denied.json())

alice 200 {'id': 'alice', 'role': 'user'}
bob 200 {'id': 'bob', 'role': 'user'}
admin 200 {'id': 'admin', 'role': 'admin'}
身份未通过： 401 {'detail': '需要有效身份'}
身份未通过： 401 {'detail': '需要有效身份'}


## 3 把资源检查放在读取之前

本例采用下列策略。管理员跨所有者操作是本例明确授予的权限，并不是 FastAPI 对 admin 这个名称的内置规则。未被规则允许的角色和操作一律拒绝。

| 当前身份 | 资源范围 | 读取、修改、删除 | 管理员入口 |
| --- | --- | --- | --- |
| 普通用户 | 自己拥有的记录 | 允许 | 拒绝 |
| 普通用户 | 他人拥有的记录 | 拒绝 | 拒绝 |
| 管理员 | 任意所有者的记录 | 允许 | 允许 |
| 未通过身份认证 | 任意记录 | 拒绝 | 拒绝 |

authorized_record 先查找记录，再检查角色和归属。身份有效但没有权限时返回 403；记录不存在时返回 404。本例保留两者的区别，便于观察。如果业务需要隐藏资源是否存在，可以对无权访问也统一返回 404。

In [4]:
def authorized_record(record_id: int, user: User) -> dict:
    record = records.get(record_id)
    if record is None:
        raise HTTPException(404, "记录不存在")
    if user.role == "admin":
        return record
    if user.role == "user" and record["owner_id"] == user.id:
        return record
    raise HTTPException(403, "无权操作这条记录")


@app.get("/records/{record_id}")
def read_record(record_id: int, user: CurrentUser) -> dict:
    return authorized_record(record_id, user)

对同一批记录分别使用两个普通用户和管理员身份发请求。权限测试要覆盖“用户与资源”的组合，不能只测一个已经登录的用户能否得到 200。

In [5]:
# 每行依次是读取 Alice、Bob、缺失记录的预期状态码。
expected_reads = {
    "alice": [200, 403, 404],
    "bob": [403, 200, 404],
    "admin": [200, 200, 404],
}
with TestClient(app) as client:
    for name, expected in expected_reads.items():
        statuses = []
        for record_id in (1, 2, 999):
            result = client.get(
                f"/records/{record_id}", headers=auth_headers[name]
            )
            statuses.append(result.status_code)
            if result.status_code == 200:
                assert result.json()["id"] == record_id
        assert statuses == expected
        print(name, statuses)

alice [200, 403, 404]
bob [403, 200, 404]
admin [200, 200, 404]


## 4 修改与删除也必须先检查

读取路由上的检查不会自动保护修改和删除路由。每种操作都先调用 authorized_record，通过以后才能改变数据。

TitleInput 只允许提交 title。Pydantic 的 extra="forbid" 会拒绝额外字段，防止请求悄悄混入 owner_id 或 role。本例只修改标题，因此不需要把整条存储记录作为输入模型。

In [6]:
from pydantic import ConfigDict, Field


class TitleInput(BaseModel):
    model_config = ConfigDict(extra="forbid")
    title: str = Field(min_length=1)


@app.patch("/records/{record_id}")
def rename_record(record_id: int, body: TitleInput, user: CurrentUser) -> dict:
    record = authorized_record(record_id, user)
    record["title"] = body.title  # 只有权限检查通过后才执行。
    return record


@app.delete("/records/{record_id}", status_code=204)
def delete_record(record_id: int, user: CurrentUser) -> Response:
    authorized_record(record_id, user)
    del records[record_id]
    return Response(status_code=204)

拒绝修改不能只检查响应中的 403，还应检查原记录的内容和数量。下面保存两个小字典的副本，再尝试双向跨用户修改和删除。

In [7]:
before_denied = {key: value.copy() for key, value in records.items()}
with TestClient(app) as client:
    for name, other_id in (("alice", 2), ("bob", 1)):
        changed = client.patch(
            f"/records/{other_id}", headers=auth_headers[name],
            json={"title": "不应写入"},
        )
        deleted = client.delete(
            f"/records/{other_id}", headers=auth_headers[name]
        )
        assert changed.status_code == deleted.status_code == 403
        assert records == before_denied
        print(name, "修改/删除他人记录：", changed.status_code, deleted.status_code)
# 本例记录只有字符串和整数，逐条 copy 足以保留比较用的原值。
print("拒绝后数据未改变：", records == before_denied)

alice 修改/删除他人记录： 403 403
bob 修改/删除他人记录： 403 403
拒绝后数据未改变： True


## 5 所有者由服务端填写

新建记录时，从当前用户取得 owner_id，客户端只提交标题。role 来自服务端用户资料，owner_id 来自服务端写入和保存的记录，两个判断依据都不接受客户端自行指定。

count(3) 提供从 3 开始的连续编号，next(record_ids) 每次取一个编号。下面的内存字典与递增编号只用于顺序请求演示；它们不提供持久化或数据库并发保证。真实存储同样需要在读取或变更目标记录时执行权限条件。

In [8]:
from itertools import count

record_ids = count(3)


@app.post("/records", status_code=201)
def create_record(body: TitleInput, user: CurrentUser) -> dict:
    if user.role not in {"user", "admin"}:
        raise HTTPException(403, "无权创建记录")
    record_id = next(record_ids)
    record = {"id": record_id, "owner_id": user.id, "title": body.title}
    records[record_id] = record
    return record

先尝试在新建和修改请求中伪造所有者或角色。这里预期的是输入校验失败 422；资源归属拒绝仍由上面的 403 路径承担。字段校验不能替代权限检查。

In [9]:
before_spoof = {key: value.copy() for key, value in records.items()}
with TestClient(app) as client:
    for method, path in (("POST", "/records"), ("PATCH", "/records/1")):
        result = client.request(
            method, path, headers=auth_headers["alice"],
            json={"title": "伪造请求", "owner_id": "bob", "role": "admin"},
        )
        assert result.status_code == 422
        rejected_fields = {
            error["loc"][-1] for error in result.json()["detail"]
        }
        assert rejected_fields == {"owner_id", "role"}
        assert records == before_spoof
        print(method, result.status_code, "数据未改变：", records == before_spoof)

POST 422 数据未改变： True
PATCH 422 数据未改变： True


再检查两个普通用户和管理员各自的正常操作：创建自己的记录，读取、修改并删除。删后重新读取应返回 404，以确认删除确实生效。

In [10]:
with TestClient(app) as client:
    for name in ("alice", "bob", "admin"):
        headers = auth_headers[name]
        created = client.post("/records", headers=headers, json={"title": "初稿"})
        assert created.status_code == 201
        assert created.json()["owner_id"] == name
        path = f"/records/{created.json()['id']}"
        read = client.get(path, headers=headers)
        changed = client.patch(path, headers=headers, json={"title": "修订稿"})
        assert read.status_code == changed.status_code == 200
        assert changed.json()["title"] == "修订稿"
        assert changed.json()["owner_id"] == name
        deleted = client.delete(path, headers=headers)
        assert deleted.status_code == 204 and deleted.content == b""
        missing = client.get(path, headers=headers)
        assert missing.status_code == 404
        statuses = [created.status_code, read.status_code, changed.status_code,
                    deleted.status_code, missing.status_code]
        print(name, "创建/读/改/删/再读：", statuses)

alice 创建/读/改/删/再读： [201, 200, 200, 204, 404]
bob 创建/读/改/删/再读： [201, 200, 200, 204, 404]
admin 创建/读/改/删/再读： [201, 200, 200, 204, 404]


## 6 用角色依赖保护管理员入口

基于角色的访问控制（RBAC）将一组权限授予角色，再给用户分配角色。只检查普通用户角色不足以区分 Alice 和 Bob 的记录，所以资源归属检查仍须保留。

require_admin 先复用身份依赖，再检查服务端角色。路由通过 Security(require_admin) 取得检查通过的用户；Security 不会根据函数名称或 admin 字符串自动判断权限。

In [11]:
def require_admin(user: CurrentUser) -> User:
    if user.role != "admin":
        raise HTTPException(403, "需要管理员权限")
    return user


@app.get("/admin/summary")
def admin_summary(user: Annotated[User, Security(require_admin)]) -> dict:
    return {"operator": user.id, "record_count": len(records)}


with TestClient(app) as client:
    for name, expected in (("alice", 403), ("bob", 403), ("admin", 200)):
        # 查询参数中的 role 没有权限来源地位，不能把普通用户变成管理员。
        result = client.get("/admin/summary?role=admin", headers=auth_headers[name])
        assert result.status_code == expected
        print(name, "管理员入口：", result.status_code)
    assert client.get("/admin/summary").status_code == 401

alice 管理员入口： 403
bob 管理员入口： 403
admin 管理员入口： 200


管理员的资源操作还要按本例策略单独检查。使用两个普通用户各自创建的新记录，管理员分别读取、修改和删除；其 owner_id 不会因管理员修改标题而改变。

In [12]:
with TestClient(app) as client:
    for owner in ("alice", "bob"):
        created = client.post(
            "/records", headers=auth_headers[owner], json={"title": "待检查"}
        )
        assert created.status_code == 201
        path = f"/records/{created.json()['id']}"
        headers = auth_headers["admin"]
        read = client.get(path, headers=headers)
        changed = client.patch(path, headers=headers, json={"title": "已检查"})
        assert read.status_code == changed.status_code == 200
        assert changed.json()["owner_id"] == owner
        assert changed.json()["title"] == "已检查"
        deleted = client.delete(path, headers=headers)
        assert deleted.status_code == 204
        assert client.get(path, headers=auth_headers[owner]).status_code == 404
        print("admin 操作", owner, "的记录：", read.status_code,
              changed.status_code, deleted.status_code)

admin 操作 alice 的记录： 200 200 204
admin 操作 bob 的记录： 200 200 204


## 7 每次请求重新检查权限

权限检查必须在服务端执行，并覆盖每次请求。前端隐藏按钮只能改变界面；已经成功访问一次，也不意味着后续操作永远获准。

下面暂时把管理员的服务端角色改为普通用户，再用同一份教学凭据发请求。此实验只说明当前依赖每次读取服务端用户资料；真实认证系统如何刷新权限、使旧令牌失效，需要由它自己的凭据和权限策略保证。

In [13]:
previous_role = users["admin"].role
before_role_change = {key: value.copy() for key, value in records.items()}
try:
    users["admin"].role = "user"
    with TestClient(app) as client:
        summary = client.get("/admin/summary", headers=auth_headers["admin"])
        changed = client.patch(
            "/records/1", headers=auth_headers["admin"], json={"title": "越权修改"}
        )
        assert summary.status_code == changed.status_code == 403
        assert records == before_role_change
        print("角色收回后：", summary.status_code, changed.status_code)
        print("数据未改变：", records == before_role_change)
finally:
    users["admin"].role = previous_role  # 还原实验资料，便于继续练习。

角色收回后： 403 403
数据未改变： True


## 8 OAuth2 scopes 表示哪些权限

OAuth2 的 scope 是表示权限的字符串，例如 records:read 表示读取记录，records:write 表示修改记录。单个 scope 不包含空格；多个 scope 在 OAuth2 协议中用空格分隔。角色是一组应用规则，scope 通常描述被授予的具体能力，两者的映射由应用设计。

使用 OAuth2 安全方案时，Security(check_scopes, scopes=["records:write"]) 可以声明路由所需权限；这里 check_scopes 表示应用自行编写的检查依赖。该依赖通过 SecurityScopes 取得当前依赖链要求的权限，仍需与已验证凭据中的授权范围进行比较。声明和 OpenAPI 展示不会替代比较代码。

下面只观察“所需权限全部包含在已授予权限中”的判断。required 是操作要求，granted 是假定已可信验证的授权集合；issubset 检查前者是否为后者的子集。这段集合操作不负责认证，也不会给上面的 HTTPBearer 应用自动增加 OAuth2 流程。

In [14]:
required = {"records:write"}
granted = {"records:read"}
print("只授予读取，能否修改：", required.issubset(granted))
assert not required.issubset(granted)

granted = {"records:read", "records:write"}
print("授予读取和修改，能否修改：", required.issubset(granted))
assert required.issubset(granted)
# 即使有 records:write，也仍须按业务规则检查要修改的具体记录归谁。

只授予读取，能否修改： False
授予读取和修改，能否修改： True


## 本章小结

（1）身份认证确定用户，授权检查用户对具体资源和操作的权限。

（2）服务端保存角色和归属，在每个读取、修改、删除入口检查；先获准，再变更数据。

（3）本例用 401 表示身份未通过，用 403 表示已识别用户权限不足，用 404 表示记录不存在；拒绝写入时还要检查原数据。

（4）Security 组织安全依赖，OAuth2 scopes 声明所需权限；实际角色、scope 与资源归属判断都需要服务端代码落实。

## 练习

（1）新增一个由 Bob 创建的记录，依次让 Alice 读取、修改和删除它。验证标准：三个请求都返回 403，Bob 仍能读取原始标题。

（2）给新建和修改请求分别只加入 owner_id 或 role。验证标准：四个请求均返回 422，记录数量、标题和归属保持原值。

（3）把策略改为“管理员可读取和修改他人记录，但不能删除他人记录”。验证标准：管理员对普通用户记录的 GET、PATCH、DELETE 分别返回 200、200、403，最后所有者仍能读取修改后的记录。

（4）为一次操作同时要求 records:read 和 records:write。验证标准：只授予其中一个 scope 时集合判断为 False，同时授予两个时为 True；说明为什么通过这个判断仍不能跳过 owner_id 检查。

提示：练习输入在本篇内自行创建。第（3）题需要让资源检查区分读取、修改和删除操作，不应只修改管理员入口；每次失败请求前后保存并比较目标记录。重新从头运行可以恢复本篇初始用户和记录。

## 参考与引用来源

- **OWASP Cheat Sheet Series**：[Authorization Cheat Sheet](https://cheatsheetseries.owasp.org/cheatsheets/Authorization_Cheat_Sheet.html) 的 Introduction、Deny by Default、Validate the Permissions on Every Request、Enforce Authorization Checks on the Right Location、Create Unit and Integration Test Cases for Authorization Logic，用于身份与授权、角色、默认拒绝、服务端逐请求检查及权限测试；[IDOR Prevention Cheat Sheet](https://cheatsheetseries.owasp.org/cheatsheets/Insecure_Direct_Object_Reference_Prevention_Cheat_Sheet.html#verifying-access-controls) 的 Verifying access controls 与 Mitigation，用于对象级归属检查和从可信身份取得当前用户。
- **FastAPI 官方文档**：[HTTPBearer 与 HTTPAuthorizationCredentials](https://fastapi.tiangolo.com/reference/security/#fastapi.security.HTTPBearer) 的 Usage、auto_error，以及 [Security 依赖](https://fastapi.tiangolo.com/reference/dependencies/#fastapi.Security)，用于读取凭据和组织安全依赖；[OAuth2 scopes](https://fastapi.tiangolo.com/advanced/security/oauth2-scopes/) 的 OAuth2 scopes and OpenAPI、Dependency tree and scopes、More details about SecurityScopes，用于权限字符串、声明与执行的区别；[Handling Errors](https://fastapi.tiangolo.com/tutorial/handling-errors/) 的 Use HTTPException 与 Add custom headers、[Testing](https://fastapi.tiangolo.com/tutorial/testing/#using-testclient)，用于错误响应和应用内请求测试。OAuth2 scopes 页仅引用 scope 与依赖部分，不采用其密码授权流程。
- **RFC Editor**：[RFC 9110 第 15.5.2–15.5.5 节](https://www.rfc-editor.org/rfc/rfc9110.html#section-15.5.2)，用于 401、403、404 和选择隐藏禁止访问资源的存在性；[RFC 6750 第 3–3.1 节](https://www.rfc-editor.org/rfc/rfc6750.html#section-3)，用于 Bearer 挑战头、无效凭据与权限不足的区别。
- **Pydantic 官方文档**：[ConfigDict.extra](https://pydantic.dev/docs/validation/latest/api/pydantic/config/#extra)，用于 extra="forbid" 拒绝输入模型未声明的字段。
- **Python 官方文档**：[Set Types](https://docs.python.org/3/library/stdtypes.html#set-types-set-frozenset) 的 issubset，用于检查权限集合的包含关系；[itertools.count](https://docs.python.org/3/library/itertools.html#itertools.count)，用于本例的连续编号。